# RCPGenerator - Getting Started

This notebook demonstrates how to generate particle packings in Python using **RCPGenerator**, a Python interface to a validated C++ random close packing engine.

RCPGenerator can generate dense packings of particles (disks, spheres, or hyperspheres) in both **flat and curved containers**, including boxes, circles, cylinders, and spheres. The package supports multiple particle size distributions and allows users to evolve packings toward **random close packing** or to **target a specific packing fraction**.

This notebook converts the example scripts in the repository into interactive sections that can be executed step-by-step. Each section demonstrates a different packing scenario and includes inline visualization so the resulting packings can be inspected directly within the notebook.

Examples covered in this notebook include:

- Disk packings in a **2D box**
- Sphere packings in a **3D box**
- Packings in **curved containers** (circle, cylinder, sphere)
- **Polydisperse particle distributions**
- **Ellipses and ellipsoids**, and **rods** with round or flat ends
- **Target packing fraction workflows**, where packings are evolved toward a desired packing fraction and then relaxed

The goal is to provide a quick introduction to the Python API and typical workflows used when generating particle packings with RCPGenerator.

Research Citation
If you use this toolkit in your research, please cite the following paper:
**Desmond, K "Random close packing at extreme size ratios with an Adam-based inflation protocol" arXiv, 2608.12235 (2026).**
[View on arXiv](https://arxiv.org/abs/2608.12235)


**Desmond, KW & Weeks, ER "Influence of particle size distribution on random close packing of spheres." Phys. Rev. E 90, 022204 (2014).**
[View on Google Scholar](https://scholar.google.com/citations?view_op=view_citation&hl=en&user=KqSFejcAAAAJ&citation_for_view=KqSFejcAAAAJ:U_HPUtbDl20C)

**Desmond, KW & Weeks, ER "Random close packing in confined geometries." Phys. Rev. E 80, 051305 (2009).**
[View on Google Scholar](https://scholar.google.com/citations?view_op=view_citation&hl=en&user=KqSFejcAAAAJ&citation_for_view=KqSFejcAAAAJ:u5HHmVD_uO8C) |

[Back to Homepage](https://kd-physics.github.io/)

[Back to repo](https://github.com/KD-physics/RCPGenerator)


Generate random close packings and dense particle packings in Python, including sphere packing in boxes and curved containers.

## Installation

#####################################################################################

This notebook begins by cloning the repository, moving into the Python package directory, and compiling the C++ core together with the Python bindings.

The installation step builds the packing engine and installs the `rcpgenerator` Python package so it can be imported directly within this notebook.

Once installation completes, the examples below will demonstrate how to generate particle packings, visualize them inline, and save figures using the Python API.

## Installing From Source

In [ ]:
!git clone https://github.com/KD-physics/RCPGenerator.git
%cd RCPGenerator/python_code/python

In [ ]:
!pip install -v .

Compilation may take a short time in Colab.

## Installing from wheels

In [ ]:
import pathlib
import subprocess
import sys
import tempfile
import urllib.request
import zipfile

VERSION = "1.2.1"

url = (
    "https://raw.githubusercontent.com/KD-physics/RCPGenerator/main/"
    "python_precompiled_wheels/rcpgenerator-Linux%20x86-64.zip"
)

folder = pathlib.Path(tempfile.mkdtemp(prefix="rcpgenerator_"))
archive = folder / "linux-wheels.zip"

print("Downloading RCPGenerator wheels...")
urllib.request.urlretrieve(url, archive)

print("Extracting wheels...")
with zipfile.ZipFile(archive) as zf:
    zf.extractall(folder)

python_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
matches = list(
    folder.rglob(
        f"rcpgenerator-{VERSION}-{python_tag}-{python_tag}-*.whl"
    )
)

assert len(matches) == 1, (
    f"Expected one RCPGenerator {VERSION} wheel for {python_tag}; "
    f"archive contains {[p.name for p in folder.rglob('*.whl')]}"
)

wheel = matches[0]
print("Installing:", wheel.name)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--force-reinstall",
    "--no-deps",
    str(wheel),
])

print("Installation complete.")

## Verify Install

In [ ]:
import rcpgenerator
print(rcpgenerator.Packing)

## Set Thread Count

In [ ]:
rcpgenerator.set_num_threads(2)

## Built-in Examples

`rcpgenerator.example()` lists short, complete snippets by topic (2D, 3D and higher-dimensional packings, every boundary and container type, arbitrary size distributions, diameter permutation, the quench, contact forces, plotting, saving). `rcpgenerator.example("name")` prints one, ready to copy, paste and modify; for example, the correct `walls` setting for a capped cylinder:

In [ ]:
rcpgenerator.example()                    # the catalogue
print()
rcpgenerator.example("capped_cylinder")   # one snippet

## Helper Functions

The following helper functions reproduce the shared utilities used by the example scripts in the repository.  
They simplify the notebook examples by handling common tasks such as printing packing summaries, displaying packings inline, and optionally saving rendered figures.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("examples/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PALETTE_MAP = {
    "2d_monodisperse_box": 1,
    "2d_bidisperse_box": 5,
    "2d_polydisperse_box": 10,
    "2d_circular_container": 12,
    "2d_powerlaw_phi_staging": 8,
    "3d_monodisperse_box": 2,
    "3d_bidisperse_box": 6,
    "3d_cylindrical_container": 9,
    "3d_spherical_container": 4,
    "starter": 7,
}

def distribution_summary(dist):
    dist_type = dist.get("type", "mono")
    if dist_type == "mono":
        return f"mono(d={dist.get('d')})"
    if dist_type == "bidisperse":
        return (
            f"bidisperse(d1={dist.get('d1')}, d2={dist.get('d2')}, "
            f"p={dist.get('p')})"
        )
    if dist_type == "lognormal":
        return f"lognormal(mu={dist.get('mu')}, sigma={dist.get('sigma')})"
    if dist_type == "flat":
        return f"flat(d_min={dist.get('d_min')}, d_max={dist.get('d_max')})"
    if dist_type == "custom":
        values = dist.get("custom", [])
        return f"custom(len={len(values)})"
    return dist_type

def walls_summary(walls):
    return "periodic" if all(int(w) == 0 for w in walls) else str(walls)

def print_case_summary(case_name, packing):
    print(f"case: {case_name}")
    print(f"N: {packing.N}")
    print(f"Ndim: {packing.Ndim}")
    print(f"box: {list(packing.box)}")
    print(f"walls: {walls_summary(list(packing.walls))}")
    print(f"distribution: {distribution_summary(dict(packing.dist))}")
    print(f"final phi: {packing.phi_final}")
    print(f"steps: {packing.steps}")
    print(f"force magnitude: {packing.force_magnitude}")
    print(f"max_min_dist: {packing.max_min_dist}")

def show_and_save(case_name, packing):
    palette_choice = PALETTE_MAP.get(case_name, 1)
    packing.show_packing(palette_choice=palette_choice)
    image_path = packing.savefig(
        path=OUTPUT_DIR / f"{case_name}.png",
        palette_choice=palette_choice,
    )
    print(f"image: {image_path}")
    return image_path

import time

def timed_pack(packing, **kwargs):
    """packing.pack(), and print how long the packing itself took. Drawing is done in a separate cell: it is
    much slower than packing and says nothing about the speed of the packer."""
    t0 = time.perf_counter()
    packing.pack(**kwargs)
    seconds = time.perf_counter() - t0
    print(f"packing time: {seconds:.2f} s   (N = {packing.N}, {packing.Ndim}D, {packing.steps} steps)")
    return seconds

class timed:
    """with timed("what"): ...   prints how long the block took."""
    def __init__(self, what):
        self.what = what
    def __enter__(self):
        self.t0 = time.perf_counter()
        return self
    def __exit__(self, *exc):
        print(f"{self.what}: {time.perf_counter() - self.t0:.2f} s")
        return False

### Packing time and drawing time

Packing is fast; drawing is not. A few hundred particles pack in about a second in 2D and a few seconds in 3D,
while Matplotlib needs several seconds to draw a 2D picture and a minute or more for a 3D one. So that the two are
not confused, every example below is split in two cells:

1. a cell that **packs** and prints `packing time: ...` (the time of `pack()` alone);
2. the **next cell**, which only **draws** the packing just made.

Run the first, read the time, then run the second.

## 2D Examples
#####################################################################################

### Simplest Setup Case

This example demonstrates the most minimal workflow using the class-based API.  
We generate a **2D monodisperse particle packing in a square box**, run the packing algorithm, and display the result inline.

In [ ]:
# Name used for printed summaries and saved output files
CASE_NAME = "2d_monodisperse_box"

# Create a packing object describing the system configuration
packing = rcpgenerator.Packing(
    phi=0.11,              # Initial packing fraction used to initialize particle sizes
    N=250,                 # Number of particles
    Ndim=2,                # Dimensionality (2D packing)
    box=[1.0, 1.0],        # Size of the simulation box
    walls=[0, 0],          # Periodic boundaries in both directions
    fix_height=False,      # Allow box height to evolve if needed
    dist={"type": "mono", "d": 1.0},  # Monodisperse particle distribution with diameter 1
    neighbor_max=0,        # Use default neighbor list settings
    seed=123,              # Random seed for reproducibility
)

# Run the packing evolution until the algorithm converges
timed_pack(packing)

# Print a short summary of the resulting packing
print_case_summary(CASE_NAME, packing)

In [ ]:
# Draw the packing made in the previous cell
# Display the packing inline and save a figure to file
show_and_save(CASE_NAME, packing)

### 2D Bidisperse Packing in a Periodic Box

This example generates a **bidisperse particle packing** in a periodic 2D box.  
Bidisperse systems contain two particle sizes, which helps prevent crystallization and produces more disordered packings.

Here we use two particle diameters (`d1` and `d2`) with equal probability.

Bidisperse systems are commonly used in packing studies because they suppress crystallization and produce more representative disordered packings.

In [ ]:
CASE_NAME = "2d_bidisperse_box"

# Create a packing configuration with two particle sizes
packing = rcpgenerator.Packing(
    phi=0.11,                    # Initial packing fraction used for initialization
    N=250,                       # Number of particles
    Ndim=2,                      # 2D packing
    box=[1.0, 1.0],              # Simulation box dimensions
    walls=[0, 0],                # Periodic boundaries in both directions
    fix_height=False,            # Allow box height evolution if required
    dist={
        "type": "bidisperse",    # Two particle sizes
        "d1": 0.8,               # Diameter of particle type 1
        "d2": 1.2,               # Diameter of particle type 2
        "p": 0.5                 # Probability of selecting type 1 (50/50 mixture)
    },
    neighbor_max=0,              # Default neighbor list behavior
    seed=124,                    # Random seed for reproducibility
)

# Run the packing algorithm
timed_pack(packing)

# Print a summary of the resulting packing
print_case_summary(CASE_NAME, packing)

In [ ]:
# Draw the packing made in the previous cell
# Save figure to disk (disabled here for notebook examples)
# show_and_save(CASE_NAME, packing)

# Display packing inline in the notebook
packing.show_packing(palette_choice=1)

### 2D Packing in a Circular Container

This example generates a **monodisperse particle packing inside a circular container**.  
The container is defined using the legacy wall convention from the original C++ implementation.

In this convention, the `walls` parameter controls the container geometry.  
Setting `walls=[-2, 0]` activates the **circular boundary condition** for a 2D packing.

In [ ]:
CASE_NAME = "2d_circular_container"

# Create a packing configuration for particles confined inside a circular container
packing = rcpgenerator.Packing(
    phi=0.25,                 # Initial packing fraction used to initialize particle sizes
    N=500,                    # Number of particles
    Ndim=2,                   # 2D packing
    box=[1.0, 1.0],           # Base box dimensions (used internally by the algorithm)
    walls=[-2, 0],            # Circular container wall convention
    fix_height=False,         # Allow box height evolution if required
    dist={"type": "mono", "d": 1.0},  # Monodisperse particles
    neighbor_max=0,           # Default neighbor list behavior
    seed=128,                 # Random seed for reproducibility
)

# Run the packing algorithm
timed_pack(packing)

# Print a short summary of the resulting packing
print_case_summary(CASE_NAME, packing)

In [ ]:
# Draw the packing made in the previous cell
# Display the packing inline and save a figure to file
# show_and_save(CASE_NAME, packing)

# Display packing
packing.show_packing(palette_choice=12)

### 2D Lognormal Polydisperse Packing in a Periodic Box

This example generates a **polydisperse particle packing** in a periodic 2D box.  
Instead of all particles having the same size, diameters are drawn from a **lognormal distribution**.

Polydisperse systems are commonly used when modeling real granular materials, emulsions, or powders where particle sizes naturally vary.

Here the particle diameters are sampled from a lognormal distribution defined by `mu` and `sigma`.

In [ ]:
CASE_NAME = "2d_polydisperse_box"

# Create a packing configuration with lognormal particle size distribution
packing = rcpgenerator.Packing(
    phi=0.11,                 # Initial packing fraction used for initialization
    N=250,                    # Number of particles
    Ndim=2,                   # 2D packing
    box=[1.0, 0.5],           # Simulation box dimensions
    walls=[0, 0],             # Periodic boundaries in both directions
    fix_height=False,         # Allow box height evolution if required
    dist={
        "type": "lognormal",  # Lognormal particle size distribution
        "mu": 0.0,            # Mean of the lognormal distribution
        "sigma": 0.35         # Width (standard deviation)
    },
    neighbor_max=0,           # Default neighbor list behavior
    seed=125,                 # Random seed for reproducibility
)

# Run the packing algorithm
timed_pack(packing)

# Print a summary of the resulting packing
print_case_summary(CASE_NAME, packing)

In [ ]:
# Draw the packing made in the previous cell
# Display the packing inline and save a figure
# show_and_save(CASE_NAME, packing)

# Display packing
packing.show_packing(palette_choice=9)

### 2D Power-Law Distribution with Target Packing Fraction Staging

This example demonstrates a **staged packing workflow**. The system first evolves toward a target packing fraction and is then incrementally compressed while relaxing particle overlaps.

Particle sizes are drawn from a **power-law distribution**, which produces a wide range of particle diameters.

The workflow proceeds as follows:

1. Evolve the packing toward an initial target packing fraction (`phi = 0.82`)
2. Increase the packing fraction step-by-step
3. After each increase, run a **fixed-diameter relaxation** to minimize overlaps

This type of staged evolution is useful when generating packings at **high packing fractions**, where directly targeting the final density may be unstable.

In [ ]:
CASE_NAME = "2d_powerlaw_phi_staging"

# Create a packing configuration using a power-law particle size distribution
packing = rcpgenerator.Packing(
    phi=0.11,                      # Initial packing fraction used for initialization
    N=250,                         # Number of particles
    Ndim=2,                        # 2D packing
    box=[1.0, 1.0],                # Simulation box dimensions
    walls=[0, 0],                  # Periodic boundaries
    fix_height=False,              # Allow box evolution if required
    dist={
        "type": "powerlaw",        # Power-law particle size distribution
        "d_min": 0.3,              # Minimum particle diameter
        "d_max": 1.8,              # Maximum particle diameter
        "exponent": -2.5           # Power-law exponent
    },
    neighbor_max=0,                # Default neighbor list behavior
    seed=131,                      # Random seed for reproducibility
)

with timed("packing time (all stages)"):
    # First evolve toward an initial target packing fraction
    packing.relax(n_steps=5000, target_phi=0.82)

    # Incrementally increase packing fraction and relax overlaps
    for target_phi in [0.84, 0.86, 0.88, 0.90, 0.92, 0.94]:
        packing.update_phi(target_phi)
        packing.relax(n_steps=1000, fix_diameter=True)

# Print summary of final packing state
print_case_summary(CASE_NAME, packing)

In [ ]:
# Draw the packing made in the previous cell
# Save figure if desired
# show_and_save(CASE_NAME, packing)

# Display packing inline in the notebook
packing.show_packing(palette_choice=11)

### A Movie of a Packing Being Made

`pack(capture_trajectory=True)` records frames while the packing runs and `animate_2d()` plays them (2D only).
Each particle keeps its colour for the whole movie.

Almost all of the motion happens in the first few hundred of some 16,000 steps: the particles spread out while
they are small, and after that they only grow in place. The frames are therefore recorded evenly in the
**logarithm** of the step (`trajectory_spacing="log"`; `trajectory_per_decade` sets how many frames per factor of
10 in steps). With the default linear spacing (`trajectory_interval`) the movie is nearly still after its first
frames. As everywhere in this notebook, the packing is timed in one cell and drawn in the next.

In [ ]:
movie_discs = rcpgenerator.Packing(
    N=150, Ndim=2, box=[1.0, 1.0], walls=[0, 0],
    dist={"type": "bidisperse", "d1": 0.8, "d2": 1.2, "p": 0.5},
    seed=7,
)
timed_pack(movie_discs, capture_trajectory=True, trajectory_spacing="log", trajectory_per_decade=12)
print(f"{len(movie_discs.trajectory_steps)} frames recorded, at steps {movie_discs.trajectory_steps[:6]} ... {movie_discs.trajectory_steps[-3:]}")

In [ ]:
# Play the frames recorded in the previous cell (building the animation is the slow part).
# To save it instead: movie_discs.animate_2d(path="packing.gif", show=False)
anim = movie_discs.animate_2d(show=True)

## 3D Examples
#####################################################################################

### 3D Bidisperse Packing in a Periodic Box

This example generates a **3D particle packing** in a periodic cubic box using a **bidisperse particle size distribution**.

Bidisperse systems contain two particle diameters and are commonly used in packing studies because they suppress crystallization and produce more disordered packings compared to monodisperse systems.

The resulting packing is rendered using the built-in 3D visualization utilities.

In [ ]:
CASE_NAME = "3d_bidisperse_box"

# Create a 3D packing configuration with two particle sizes
packing = rcpgenerator.Packing(
    phi=0.08,                 # Initial packing fraction used for initialization
    N=250,                    # Number of particles
    Ndim=3,                   # 3D packing
    box=[1.0, 1.0, 1.0],      # Simulation box dimensions
    walls=[0, 0, 0],          # Periodic boundaries in all directions
    fix_height=False,         # Allow box evolution if required
    dist={
        "type": "bidisperse", # Two particle sizes
        "d1": 0.85,           # Diameter of particle type 1
        "d2": 1.15,           # Diameter of particle type 2
        "p": 0.5              # Probability of selecting type 1
    },
    neighbor_max=0,           # Default neighbor list behavior
    seed=127,                 # Random seed for reproducibility
)

# Run the packing algorithm
timed_pack(packing)

# Print summary information about the packing
print_case_summary(CASE_NAME, packing)

In [ ]:
# Draw the packing made in the previous cell
# Display and save a rendered figure of the packing
# show_and_save(CASE_NAME, packing)

# Display packing
packing.show_packing()

### 3D Packing in a Cylindrical Container

This example generates a **monodisperse particle packing inside a cylindrical container**.  
The container is defined using the legacy wall convention inherited from the original C++ implementation.

In this convention, the `walls` parameter controls the container geometry.  
Setting `walls=[-2, 0, 0]` activates the **cylindrical boundary condition** for a 3D packing.

Particles evolve inside the cylinder until the packing algorithm converges.

In [ ]:
CASE_NAME = "3d_cylindrical_container"

# Create a packing configuration for particles confined inside a cylinder
packing = rcpgenerator.Packing(
    phi=0.25,                 # Initial packing fraction used for initialization
    N=500,                    # Number of particles
    Ndim=3,                   # 3D packing
    box=[1.0, 1.0, 1.0],      # Base box dimensions used internally
    walls=[-2, 0, 0],         # Cylindrical container wall convention
    fix_height=False,         # Allow box height evolution if required
    dist={"type": "mono", "d": 1.0},  # Monodisperse particles
    neighbor_max=0,           # Default neighbor list behavior
    seed=129,                 # Random seed for reproducibility
)

# Run the packing algorithm
timed_pack(packing)

# Print summary information about the resulting packing
print_case_summary(CASE_NAME, packing)

In [ ]:
# Draw the packing made in the previous cell
# Save figure if desired
# show_and_save(CASE_NAME, packing)

# Display packing inline in the notebook
packing.show_packing()

### 3D Monodisperse Packing in a Periodic Box

This example generates a **3D monodisperse particle packing** in a periodic cubic box.  
All particles have the same diameter, and periodic boundary conditions are applied in all three spatial directions.

Monodisperse systems are the simplest packing scenario and provide a baseline for studying random close packing behavior in three dimensions.

In [ ]:
CASE_NAME = "3d_monodisperse_box"

# Create a 3D packing configuration with identical particle sizes
packing = rcpgenerator.Packing(
    phi=0.08,                 # Initial packing fraction used for initialization
    N=250,                    # Number of particles
    Ndim=3,                   # 3D packing
    box=[1.0, 1.0, 1.0],      # Simulation box dimensions
    walls=[0, 0, 0],          # Periodic boundaries in all directions
    fix_height=False,         # Allow box evolution if required
    dist={"type": "mono", "d": 1.0},  # Monodisperse particle distribution
    neighbor_max=0,           # Default neighbor list behavior
    seed=126,                 # Random seed for reproducibility
)

# Run the packing algorithm
timed_pack(packing)

# Print summary information about the resulting packing
print_case_summary(CASE_NAME, packing)

In [ ]:
# Draw the packing made in the previous cell
# Display and save the rendered packing
# show_and_save(CASE_NAME, packing)

# Display packing
packing.show_packing(palette_choice= 5)

### 3D Packing in a Spherical Container

This example generates a **monodisperse particle packing inside a spherical container**.  
The container geometry is defined using the legacy wall convention inherited from the original C++ implementation.

In this convention, the `walls` parameter determines the container type.  
Setting `walls=[-3, 0, 0]` activates the **spherical boundary condition** for a 3D packing.

Particles evolve inside the spherical domain until the packing algorithm converges.

In [ ]:
CASE_NAME = "3d_spherical_container"

# Create a packing configuration for particles confined inside a sphere
packing = rcpgenerator.Packing(
    phi=0.25,                 # Initial packing fraction used for initialization
    N=500,                    # Number of particles
    Ndim=3,                   # 3D packing
    box=[1.0, 1.0, 1.0],      # Base box dimensions used internally
    walls=[-3, 0, 0],         # Spherical container wall convention
    fix_height=False,         # Allow box height evolution if required
    dist={"type": "mono", "d": 1.0},  # Monodisperse particle distribution
    neighbor_max=0,           # Default neighbor list behavior
    seed=130,                 # Random seed for reproducibility
)

# Run the packing algorithm
timed_pack(packing)

# Print summary information about the resulting packing
print_case_summary(CASE_NAME, packing)

In [ ]:
# Draw the packing made in the previous cell
# Display and save the rendered packing
# show_and_save(CASE_NAME, packing)

# Display packing
packing.show_packing()

## Ellipsoids
#####################################################################################

Pass `aspect_ratios` to pack ellipses (2D) or ellipsoids (3D and above) instead of discs and spheres.

- `D` is the full length of a particle along its **first** body axis, which is its **longest** axis.
- `aspect_ratios` holds the other `Ndim - 1` axes in units of `D`, written in **decreasing order**, each at most 1:
  `[0.5]` in 2D is a 2:1 ellipse; `[0.8, 0.6]` in 3D has axes `D`, `0.8 D`, `0.6 D`.
- One list applies to every particle. A list of `N` rows gives each particle its own shape.
- Orientations are generated at random and evolve with the positions. After `pack()`, `packing.orientations` holds
  one rotation matrix per particle (columns = body axes).
- `packing.phi_contact` is the packing fraction after shrinking to exact contact (no overlap);
  `packing.phi_final` is the raw value at the end of the run, as for spheres.

Without `aspect_ratios` nothing changes: the sphere code runs as before.

### 2D Ellipses in a Periodic Box

In [ ]:
packing = rcpgenerator.Packing(
    N=300,                    # Number of particles
    Ndim=2,                   # 2D packing
    box=[1.0, 1.0],           # Simulation box dimensions
    walls=[0, 0],             # Periodic boundaries
    aspect_ratios=[0.5],      # second axis = 0.5 D: 2:1 ellipses
    seed=1,
)
timed_pack(packing)
print(f"phi at contact: {packing.phi_contact:.4f}")
ellipses_2d = packing

In [ ]:
# Draw the packing made in the previous cell
ellipses_2d.show_packing(palette_choice=1)

### 3D Ellipsoids

Three different axes: `D`, `0.8 D` and `0.6 D`. Drawing a 3D packing takes much longer than packing it.

In [ ]:
packing = rcpgenerator.Packing(
    N=250, Ndim=3, box=[1.0, 1.0, 1.0], walls=[0, 0, 0],
    aspect_ratios=[0.8, 0.6],        # axes D, 0.8 D, 0.6 D
    seed=1,
)
timed_pack(packing)
print(f"phi at contact: {packing.phi_contact:.4f}")
ellipsoids_3d = packing

In [ ]:
# Draw the packing made in the previous cell (slow: a minute or so)
ellipsoids_3d.show_packing(palette_choice=2)

### A Shape per Particle, and a Size Distribution

`aspect_ratios` may be a list of `N` rows, one shape per particle. With generated diameters, as here, row `i`
goes to the `i`-th largest particle; with your own diameter list (`dist={"type": "custom", ...}`) row `i` goes
with entry `i` of that list.
Here half the particles are nearly round and half are 2.5:1, with lognormal sizes.

In [ ]:
N = 300
shapes = [[0.9]] * (N // 2) + [[0.4]] * (N - N // 2)      # one row per particle
packing = rcpgenerator.Packing(
    N=N, Ndim=2, box=[1.0, 1.0], walls=[0, 0],
    dist={"type": "lognormal", "mu": 0.0, "sigma": 0.25},
    aspect_ratios=shapes,
    seed=3,
)
timed_pack(packing)
print(f"phi at contact: {packing.phi_contact:.4f}")
ellipses_mixed = packing

In [ ]:
# Draw the packing made in the previous cell
ellipses_mixed.show_packing(palette_choice=10)

### Quenching Ellipsoids

`quench()` brings an ellipsoid packing to force **and torque** balance. `quench()` does not expand the diameters
unless `kappa_mult` is passed, and a packing straight from `pack()` needs a small expansion to carry load
(without one the quench usually ends as "unjammed"). For ellipsoids use `kappa_mult=1.01` (spheres in 2D and
3D: 1.003). The largest time step `dt_max` defaults to 0.2 instead of the 0.5 of spheres, because elongated
particles need a gentler minimiser.

**If a quench stalls or fails, retry it from the packed state** with either a lower `dt_max` (half the default,
say) or slightly more compression (`kappa_mult`). A measured example: 400 ellipses of elongation 4
(`aspect_ratios=[0.25]`) that were still at a residual of 0.4 after 200,000 iterations at `kappa_mult=1.01` and
the default `dt_max`:

| retry of the packed state | outcome | iterations |
|---|---|---|
| `quench(kappa_mult=1.01, dt_max=0.1)` | converged | 35,248 |
| `quench(kappa_mult=1.01, dt_max=0.05)` | converged | 157,347 |
| `quench(kappa_mult=1.02)` | converged | 15,830 |
| the same settings, then `quench(resume=True)` | not converged | 200,000 more |

Either remedy works. Going much lower on `dt_max` than needed costs many more iterations (4.5 times as many at
0.05 as at 0.1), and resuming with unchanged settings does not help a quench that is stuck.

In [ ]:
packed = ellipses_2d.copy()
with timed("quench time"):
    packed.quench(kappa_mult=1.01)      # expand the diameters by 1 %, then balance
print(packed.quench_status, f"| residual = {packed.quench_residual:.1e} | iterations = {packed.quench_iters}")

# What to do when a quench does not converge (this one did; the retry is shown for the pattern):
# start again from the PACKED state with a lower dt_max, or with slightly more compression (kappa_mult=1.02)
retry = ellipses_2d.copy()
with timed("quench time (dt_max = 0.1)"):
    retry.quench(kappa_mult=1.01, dt_max=0.1)
print(retry.quench_status, f"| residual = {retry.quench_residual:.1e} | iterations = {retry.quench_iters}")

## Rods
#####################################################################################

Two kinds of rod, chosen with `rod_capped`:

- **capped** (`rod_capped=True`): a straight section with a half-sphere on each end (2D: a stadium);
- **uncapped** (`rod_capped=False`): flat ends: a cylinder (2D: a rectangle).

The shape is given by two numbers in units of the particle's `D`:

- `rod_length`: the length of the straight section, `D * rod_length`;
- `rod_radius`: the radius, `D * rod_radius`.

With `rod_radius=0.5`, `D` is the thickness of the rod and `rod_length` is its straight length in thicknesses; a
capped rod is then `rod_length + 1` thicknesses from tip to tip. Each of the two may be one number or a list of
`N`. As for ellipsoids, orientations are generated and returned (the rod's axis is the first column of its
rotation matrix), and `phi_contact` is the packing fraction at exact contact. Uncapped rods are available in 2D
and 3D.

### 2D Capped Rods

In [ ]:
packing = rcpgenerator.Packing(
    N=300, Ndim=2, box=[1.0, 1.0], walls=[0, 0],
    rod_length=2.0,           # straight section: 2 thicknesses
    rod_radius=0.5,           # radius 0.5 D, so D is the thickness
    rod_capped=True,          # round ends
    seed=1,
)
timed_pack(packing)
print(f"phi at contact: {packing.phi_contact:.4f}")
rods_2d_capped = packing

In [ ]:
# Draw the packing made in the previous cell
rods_2d_capped.show_packing(palette_choice=1)

### 2D Uncapped Rods (Rectangles) with Hard Walls Top and Bottom

In [ ]:
packing = rcpgenerator.Packing(
    N=300, Ndim=2, box=[1.0, 1.0],
    walls=[0, 1],             # periodic in x, hard walls in y
    rod_length=2.0, rod_radius=0.5,
    rod_capped=False,         # flat ends: 2 x 1 rectangles
    seed=2,
)
timed_pack(packing)
print(f"phi at contact: {packing.phi_contact:.4f}")
rods_2d_flat = packing

In [ ]:
# Draw the packing made in the previous cell
rods_2d_flat.show_packing(palette_choice=5)

### Two Rod Lengths in One Packing

`rod_length` (and `rod_radius`) may be lists with one entry per particle.

In [ ]:
N = 300
packing = rcpgenerator.Packing(
    N=N, Ndim=2, box=[1.0, 1.0], walls=[0, 0],
    rod_length=[0.5] * (N // 2) + [3.0] * (N - N // 2),   # short and long rods of one thickness
    rod_radius=0.5, rod_capped=True, seed=1,
)
timed_pack(packing)
print(f"phi at contact: {packing.phi_contact:.4f}")
rods_mixed = packing

In [ ]:
# Draw the packing made in the previous cell
rods_mixed.show_packing(palette_choice=6)

### 3D Rods

Capped rods (spherocylinders) and flat-ended cylinders. Flat-ended rods in 3D are the slowest shape to pack: two
flat faces resting on each other have no closed-form contact, so each such contact is solved iteratively.

In [ ]:
rods_3d = {}
for capped in (True, False):
    packing = rcpgenerator.Packing(
        N=250, Ndim=3, box=[1.0, 1.0, 1.0], walls=[0, 0, 0],
        rod_length=1.5, rod_radius=0.5, rod_capped=capped, seed=1,
    )
    print("capped rods:" if capped else "flat-ended rods:")
    timed_pack(packing)
    print(f"phi at contact: {packing.phi_contact:.4f}")
    rods_3d[capped] = packing

In [ ]:
# Draw the capped-rod packing made in the previous cell (slow: a minute or so)
rods_3d[True].show_packing(palette_choice=4)

In [ ]:
# Draw the flat-ended packing made two cells above (slow: a minute or so)
rods_3d[False].show_packing(palette_choice=9)

### Ellipsoids and Rods in Curved Containers

Ellipsoids and rods pack wherever spheres do, with the same `walls` values: `[-2, 0]` a disk, `[-2, 0, 0]` a
cylinder, `[-3, 0, 0]` a sphere. `phi_contact` is measured against the container's own volume.

In [ ]:
containers = {}
for name, kw in {"oblate ellipsoids in a sphere": dict(Ndim=3, walls=[-3, 0, 0], aspect_ratios=[1.0, 0.5]),
                 "rectangles in a disk": dict(Ndim=2, walls=[-2, 0], rod_length=2.0, rod_radius=0.5, rod_capped=False)}.items():
    packing = rcpgenerator.Packing(N=250, box=[1.0] * kw["Ndim"], seed=1, **kw)
    print(name + ":")
    timed_pack(packing)
    print(f"phi at contact: {packing.phi_contact:.4f}")
    containers[name] = packing

In [ ]:
# Draw the two packings made in the previous cell (the 3D one is the slow one)
containers["rectangles in a disk"].show_packing(palette_choice=10)
containers["oblate ellipsoids in a sphere"].show_packing(palette_choice=4)

### Quenching Rods

`quench()` works for rods in 2D and 3D. A rod with sharp edges has no well-defined contact force where two flat
sides or ends rest on each other, so for the quench each rod is replaced by a very slightly **rounded** rod
(`rod_round=0.01`, 1 %), which gives a smooth energy and a stable minimisation. The packing itself is made with
the sharp shape. As for ellipsoids, pass `kappa_mult=1.01`: without an expansion the quench usually ends as
"unjammed".

As for every shape: **if the quench stalls or fails, quench the packed state again with a lower `dt_max` or
slightly more compression (`kappa_mult`)**; see the table in the ellipsoid section above.

**Flat-ended rods in 3D inside a curved container (cylinder, sphere) often need that retry.** A flat end standing
on a curved wall is held by many points of its rim at once, a much stiffer contact than the others. In a survey
of 150 such rods, three seeds per container, at `kappa_mult=1.01`: a periodic box converged 3 of 3 at the default
`dt_max`; a sphere 1 of 3, and 3 of 3 with `dt_max=0.1`; a cylinder 2 of 3, where the third did not converge at `dt_max=0.05`
either and did with `quench(kappa_mult=1.02)`. So: first a lower `dt_max`; if that does not help, slightly more
compression.

In [ ]:
quenched = rods_2d_capped.copy()
with timed("quench time"):
    quenched.quench(kappa_mult=1.01)
print(quenched.quench_status, f"| residual = {quenched.quench_residual:.1e} | iterations = {quenched.quench_iters}")

### A Movie of Rods Being Packed

`pack(capture_trajectory=True)` records frames; `animate_2d()` plays them, with every rod drawn at its recorded
position, size and orientation, and each particle keeping its colour. Discs and ellipses work the same way.

Almost all of the motion happens in the first few hundred of some 16,000 steps, so the frames are recorded
evenly in the **logarithm** of the step (`trajectory_spacing="log"`, `trajectory_per_decade` frames per factor
of 10 in steps). With the default linear spacing the movie would be nearly still after its first frames.

In [ ]:
movie = rcpgenerator.Packing(N=120, Ndim=2, box=[1.0, 1.0], walls=[0, 0],
                             rod_length=2.0, rod_radius=0.5, rod_capped=True, seed=4)
timed_pack(movie, capture_trajectory=True, trajectory_spacing="log", trajectory_per_decade=12)
print(f"{len(movie.trajectory_steps)} frames recorded")

In [ ]:
# Play the frames recorded in the previous cell (building the animation is the slow part)
anim = movie.animate_2d(show=True)

## Diameter Permutation and Quench to Force Balance
#####################################################################################

Version 1.1 adds two optional features. Both are off by default, and with them off the package runs the original algorithm bit for bit.

- **Diameter permutation** (`diameter_permute=True`) reassigns *which particle carries which diameter* while the packing is compressed. Particles are not moved and the set of diameters is unchanged; small particles end up in the voids the large ones leave, so the packing jams denser.
- **Quench to force balance** (`packing.quench()`) drives a finished packing to mechanical equilibrium: every particle's contact forces cancel.

### Diameter Permutation

The same particles are packed three times: permutation off, on at a low budget, and on at the default setting. `permute_budget` (the cap on exchanges per pass, as a fraction of N) is the main dial: it moves the result smoothly from the original algorithm to the full gain at the default of 0.25. The packing fraction printed is the value at the end of the run with the residual overlap removed. This cell takes a minute or two.

In [ ]:
import numpy as np
import rcpgenerator

def powerlaw_diameters(n, size_ratio, exponent=-3.0, seed=0):
    """n deterministic diameters with number density ~ D**exponent between 1 and size_ratio."""
    a = exponent + 1.0
    u = (np.arange(n) + 0.5) / n
    d = (1.0 + u * (size_ratio ** a - 1.0)) ** (1.0 / a)
    return np.random.default_rng(seed).permutation(d).tolist()

def phi_estimate(packing):
    """Packing fraction at the end of the run with the residual overlap removed."""
    return packing.phi_history[-1] * (1.0 - packing.max_overlap_history[-1]) ** packing.Ndim

def build(**permute_kwargs):
    return rcpgenerator.Packing(
        phi=0.05, N=2000, Ndim=3, box=[1.0, 1.0, 1.0], walls=[0, 0, 0],
        dist={"type": "custom", "custom": powerlaw_diameters(2000, 10.0)},
        seed=1, **permute_kwargs,
    )

for label, kwargs in [("off (original algorithm)", {}),
                      ("on, budget 0.01", {"diameter_permute": True, "permute_budget": 0.01}),
                      ("on, default (budget 0.25)", {"diameter_permute": True})]:
    packing = build(**kwargs)
    timed_pack(packing)
    print(f"{label:<28} phi = {phi_estimate(packing):.4f}   exchanges = {packing.permute_exchanges}")

### Quench to Force Balance

A packing run stops at the edge of jamming, where the particles barely touch. `quench(kappa_mult=1.003)` first multiplies every diameter once by `kappa_mult` (here 0.3 % larger; use 1.01 in 4D and above and for ellipsoids and rods) so that a load-bearing contact network exists, then minimises the overlap energy (FIRE) until every load-bearing particle's net force is below `tol` (default 1e-10) of its contact-force sum. The packing is changed in place and particle `i` keeps index `i`. `resume=True` continues from a previous quench without rescaling again, so a quench can be tightened in stages. `quench()` does not expand the diameters unless `kappa_mult` is passed; without it, the quench of a packing straight from `pack()` usually ends as "unjammed".

In [ ]:
packed = packing.copy()            # the permuted packing from the cell above; keep a copy

packing.quench(kappa_mult=1.003)   # diameters x 1.003 once, then FIRE to a residual of 1e-10
print(packing.quench_status, "|", packing.quench_message)

# Tighten in stages: a quick pass, then continue from where it stopped (no second rescaling).
staged = packed.copy()
staged.quench(tol=1e-4, kappa_mult=1.003)
staged.quench(tol=1e-10, resume=True)
print(staged.quench_status, f"residual = {staged.quench_residual:.2e}")

### When the Quench Stops Early

`quench()` never runs on indefinitely. When it cannot finish (iteration cap reached, no further progress, no load-bearing contacts, a collapsing time step) it stops, keeps the state it reached, raises a `RuntimeWarning`, and explains the outcome and what to try next in `quench_status` and `quench_message`. Here the iteration cap is deliberately too low; `resume=True` then continues from where it stopped, so nothing is lost.

At large size ratios (above about 20) the quench often stops for a real reason: most particles are rattlers and the load runs through small particles with only two or three contacts. The README's *Quench to force balance* section explains the options.

**When a quench stalls or fails, retry it from the packed state** with either a lower `dt_max` or slightly more
compression (`kappa_mult`). `dt_max` is the largest time step the minimiser may take (default 0.5 for spheres,
0.2 for ellipsoids and rods); `kappa_mult` is the factor applied once to every diameter before the minimisation
(default 1.0, no expansion; recommended 1.003 for spheres in 2D and 3D, 1.01 in 4D and above and for ellipsoids and
rods).
Keep a copy of the packed state for this, because a quench changes the packing in place. Do not go much lower
on `dt_max` than needed (it costs many more iterations), and do not expect `resume=True` with unchanged
settings to rescue a quench that is stuck: it is for a run that was still improving when it stopped. The
ellipsoid section above has a measured example.

In [ ]:
import warnings

short = packed.copy()                  # the permuted packing again
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # the message is printed below instead
    short.quench(max_iter=500, kappa_mult=1.003)   # deliberately too few iterations
print(short.quench_status, "|", short.quench_message)

short.quench(resume=True)              # continue from where it stopped
print(short.quench_status, f"residual = {short.quench_residual:.2e}")

## A Denser Packing, Resolved Contacts, and a Quench That Leaves the Diameters Alone

Three tools that are used one after the other:

| Goal | Call |
|---|---|
| A denser packing | `pack(extend=...)` |
| Force balance, with the contact network resolved | `adaptive_quench()` |
| Force balance at fixed diameters (after an expansion, or after you moved particles) | `quench()` |

### A Longer First Stage: `extend`

A packing run has two parts. In the first, the pressure term `mu` that swells the particles is held at its
starting value and the packing compacts. In the second, `mu` is lowered and the overlaps are worked out.
`extend=x` makes the first part `x` times as long and leaves the second part as it is. A longer first part gives a
denser packing and costs the added steps.

In [ ]:
import rcpgenerator

settings = dict(N=2000, Ndim=3, phi=0.05, box=[1.0, 1.0, 1.0], walls=[0, 0, 0],
                dist={"type": "powerlaw", "d_min": 1.0, "d_max": 5.0, "exponent": -3.0}, seed=1)

standard = rcpgenerator.Packing(**settings)
timed_pack(standard)

longer = rcpgenerator.Packing(**settings)
timed_pack(longer, extend=3)                 # first part three times as long; the same as Packing(..., extend=3)

print(f"standard : {standard.steps} steps, packing fraction at contact {standard.phi_contact:.5f}")
print(f"extend=3 : {longer.steps} steps, packing fraction at contact {longer.phi_contact:.5f}")

### Adaptive Quench: Force Balance with the Diameters Free

A packing from `pack()` is close to contact but its forces do not cancel, so which particles really touch is not
settled. `adaptive_quench()` settles it. It balances the forces while one common scale factor on all diameters
follows the pressure `mu`: the particles swell while their contacts carry less than `mu` and shrink while they carry
more. The default `mu` is `packing.mu_final`, the value the packing run ended on. A larger `mu` gives a more
compressed packing. Spheres only.

After a run or a quench the contact numbers are on the packing: `z` (contacts per particle, a wall counting as one),
`z_mean`, `n_rattlers` (particles with fewer than `Ndim + 1` contacts, which cannot be held in place) and
`z_mean_no_rattlers`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

packing = longer.copy()
print(f"after pack()            : z mean {packing.z_mean:.3f}, {packing.n_rattlers} rattlers, "
      f"{packing.z_mean_no_rattlers:.3f} without them   (not in balance yet)")
z_packed = np.array(packing.z)

with timed("adaptive quench"):
    packing.adaptive_quench()                # mu = packing.mu_final; tol=1e-9 on the global force residual
print(packing.quench_message)
print(f"after adaptive_quench() : z mean {packing.z_mean:.3f}, {packing.n_rattlers} rattlers, "
      f"{packing.z_mean_no_rattlers:.3f} without them   (2 x Ndim = {2 * packing.Ndim})")
print(f"packing fraction {packing.phi_final:.5f}, at contact {packing.phi_contact:.5f}, mu {packing.mu_final:.2e}")

bins = np.arange(-0.5, max(z_packed.max(), max(packing.z)) + 1.5)
plt.hist(z_packed, bins=bins, alpha=0.5, label="after pack()")
plt.hist(packing.z, bins=bins, alpha=0.5, label="after adaptive_quench()")
plt.xlabel("contacts per particle, z"); plt.ylabel("particles"); plt.legend(); plt.show()

### Quench at Fixed Diameters

`quench()` balances the forces with the diameters frozen. It does not expand the diameters unless `kappa_mult` is
passed, so it can be called again and again, for example after each change you make to a packing, without the
particles growing.

- A state left by `adaptive_quench()` carries load already: `quench()` needs no expansion there. At the small
  overlaps of such a state the tightest tolerance it can reach is about 1e-9, so use `tol=1e-8`.
- A packing straight from `pack()` needs a small expansion to carry load: `quench(kappa_mult=1.003)` for spheres in
  2D and 3D, `1.01` in 4D and above and for ellipsoids and rods. Without one the quench usually ends as "unjammed".

In [ ]:
balanced = packing.copy()
diameters_before = np.array(balanced.diameters)
balanced.quench(tol=1e-8)                    # no expansion: the diameters stay exactly as they are
print(balanced.quench_message)
print("diameters unchanged:", np.array_equal(diameters_before, np.array(balanced.diameters)))

compressed = longer.copy()                   # the packed state, expanded by 0.3 % and balanced
compressed.quench(kappa_mult=1.003)
print(compressed.quench_message)
print(f"z mean without rattlers: {compressed.z_mean_no_rattlers:.3f}, "
      f"packing fraction {compressed.phi_final:.5f}")

### Deforming a Balanced Packing: Hold, Move, Quench

Shear is done by you: hold some particles, move them, and quench. Here a packing between two walls is balanced,
the particles touching the top wall are moved sideways in small steps, and after each step `quench(held=...)`
restores the balance of all the others. The diameters never change. The overlap energy stored in the packing
(the fourth entry of each record in `quench_log`) shows it being strained.

In [ ]:
slab = rcpgenerator.Packing(N=1500, Ndim=3, phi=0.05, box=[1.0, 1.0, 1.0], walls=[0, 0, 1],
                            dist={"type": "bidisperse", "d_small": 1.0, "d_large": 1.4, "fraction_small": 0.5}, seed=2)
timed_pack(slab)
slab.adaptive_quench(mu=10 * slab.mu_final)  # a firmer packing: ten times the pressure the run ended on
slab.quench(tol=1e-7)
print(f"before shearing: overlap energy {slab.quench_log[-3]:.4e}; z without rattlers {slab.z_mean_no_rattlers:.3f}")

x = np.array(slab.positions); r = 0.5 * np.array(slab.diameters)
bottom = np.where(x[:, 2] < r)[0]            # particles touching the bottom wall
top = np.where(x[:, 2] > slab.box[2] - r)[0] # particles touching the top wall
held = np.concatenate([bottom, top]).tolist()
step = 0.02 * r.mean()                       # each step moves the top layer by 2 % of a mean radius

for k in range(1, 4):
    x = np.array(slab.positions)
    x[top, 0] = (x[top, 0] + step) % slab.box[0]
    slab.positions = x.tolist()
    slab.quench(held=held, tol=1e-7)
    energy = slab.quench_log[-3]              # records are (iteration, max residual, global residual, energy, count, dt)
    print(f"step {k}: {slab.quench_status} in {slab.quench_iters} iterations; "
          f"overlap energy {energy:.4e}; z without rattlers {slab.z_mean_no_rattlers:.3f}")

## Higher Dimensions
#####################################################################################

The packing core works in any dimension `Ndim >= 2` (the built-in rendering is 2D and 3D only), and so do diameter permutation and the quench. Two things differ above 3D:

- The quench's recommended compression is `kappa_mult=1.01` (1.003 in 2D and 3D): broad size distributions in 4D and above need it to form a load-bearing contact network.
- A periodic box must be wide compared with the largest particle: in 4D and above about 2.5 to 3 largest diameters, depending on the size distribution. Few particles with a broad size range in a high dimension can fall below that, and the package then warns after the run. Use more particles, or hard walls (`walls` entry `1`).

The cells below pack 4D (periodic) and 5D (hard walls) particles with a size ratio of 5, with and without permutation, and quench them. Together they take about two minutes.

In [ ]:
# 4D, periodic box, size ratio 5: permutation off and on
d4 = powerlaw_diameters(2000, 5.0)
packings_4d = {}
for label, kwargs in [("off", {}), ("permute", {"diameter_permute": True})]:
    p = rcpgenerator.Packing(phi=0.1, N=2000, Ndim=4, box=[1.0] * 4, walls=[0] * 4,
                             dist={"type": "custom", "custom": d4}, seed=1, **kwargs)
    timed_pack(p)
    packings_4d[label] = p
    print(f"4D {label:<8} phi = {phi_estimate(p):.4f}   exchanges = {p.permute_exchanges}")

### Passing One Quench to the Next

The result of a quench is an ordinary `Packing`: it can be copied, saved, reloaded and quenched further. Here the permuted 4D packing gets a quick first pass, is saved as a plain dictionary with `to_dict()` (ready for `json` or `pickle`), reloaded with `Packing.from_dict()`, and the reloaded packing finishes the quench with `resume=True` (no second rescaling). The two passes together cost about the same as one full quench. The unpermuted packing is quenched in one pass for comparison.

In [ ]:
import json

p4 = packings_4d["permute"].copy()
p4.quench(tol=1e-4, kappa_mult=1.01)                 # quick first pass (diameters x 1.01 once)
print("pass 1:", p4.quench_status, f"residual = {p4.quench_residual:.1e}", f"iterations = {p4.quench_iters}")

saved = json.dumps(p4.to_dict())                     # e.g. write this to a file
p4 = rcpgenerator.Packing.from_dict(json.loads(saved))
p4.quench(tol=1e-10, resume=True)                    # continue: no second rescaling
print("pass 2:", p4.quench_status, f"residual = {p4.quench_residual:.1e}", f"iterations = {p4.quench_iters}")

off4 = packings_4d["off"].copy()
off4.quench(kappa_mult=1.01)
print("no permutation, one pass:", off4.quench_status, f"iterations = {off4.quench_iters}")

### 5D with Hard Walls

With a thousand particles in 5D a periodic box would be too narrow for the largest particles, so this example uses hard walls in every direction (`walls=[1] * 5`). The packing fraction is low here because, in a box only about two large diameters wide, much of the volume lies against the walls.

In [ ]:
# 5D, hard walls, size ratio 5: permutation off and on, each quenched
d5 = powerlaw_diameters(1000, 5.0)
for label, kwargs in [("off", {}), ("permute", {"diameter_permute": True})]:
    p = rcpgenerator.Packing(phi=0.05, N=1000, Ndim=5, box=[1.0] * 5, walls=[1] * 5,
                             dist={"type": "custom", "custom": d5}, seed=1, **kwargs)
    timed_pack(p)
    phi_packed = phi_estimate(p)
    p.quench(kappa_mult=1.01)
    print(f"5D {label:<8} phi = {phi_packed:.4f}   quench: {p.quench_status} after {p.quench_iters} iterations")

See the README sections *Diameter permutation* and *Quench to force balance* for every setting, the measured gains at large size ratios, and holding particles for quasi-static shear.

## A Ray-Traced Picture (Optional)

The pictures above are drawn with Matplotlib. For a 3D packing a ray tracer gives a much better picture: correct
overlaps, shadows, and a container you can see through. This last example installs the free ray tracer
[Mitsuba 3](https://www.mitsuba-renderer.org/), packs 750 oblate ellipsoids in a sphere and renders them, and then does the same for spheres in a box. The renderer draws spheres or ellipsoids, in a box or in a spherical container.

- **GPU or CPU.** The renderer checks whether this session has a GPU attached that Mitsuba can use, and falls back
  to the CPU otherwise; it prints which one it used. In Colab a GPU is attached under *Runtime > Change runtime
  type*.
- **Quality.** `pixels` and `samples` are set low so that the example is quick. Raise them for a cleaner and
  larger picture (for example `pixels=1400, samples=256`); the time grows in proportion to `pixels**2 * samples`.

In [ ]:
%pip install -q mitsuba

In [ ]:
#@title The ray tracer: run this cell once (its code is hidden; open it to read or change it)
import os, shutil, subprocess, sys, time
import numpy as np

CANDY = [(0.78, 0.05, 0.07), (0.95, 0.42, 0.05), (0.98, 0.80, 0.08), (0.10, 0.55, 0.16), (0.05, 0.25, 0.70), (0.28, 0.14, 0.07)]


def gpu_renders():
    # True if a GPU is attached AND Mitsuba's GPU variant renders on it. Tested in a separate Python process,
    # because a GPU start that goes wrong can end the process instead of raising an error.
    if shutil.which("nvidia-smi") is None or subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
        return False
    test = ("import mitsuba as mi, numpy as np\n"
            "mi.set_variant('cuda_ad_rgb')\n"
            "img = np.array(mi.render(mi.load_dict(mi.cornell_box()), spp=1))\n"
            "print('GPU-OK' if np.isfinite(img).all() and img.mean() > 0 else 'GPU-BAD')\n")
    try:
        r = subprocess.run([sys.executable, "-c", test], capture_output=True, text=True, timeout=300)
    except Exception:
        return False
    return r.returncode == 0 and "GPU-OK" in r.stdout


def render_raytraced(packing, pixels=600, samples=32, path="packing_raytraced.png", use_gpu=None, seed=3):
    # Ray-traced picture of a 3D packing: spheres or ellipsoids, in a box or in a spherical container.
    #   pixels    width and height of the picture
    #   samples   rays per pixel: more gives a smoother picture and takes proportionally longer
    #   use_gpu   None: use the GPU if one is attached and works, else the CPU. True / False: force it.
    # Returns the picture (a PIL image, transparent background) and saves it to `path`.
    import time
    import numpy as np
    import mitsuba as mi
    from PIL import Image
    if use_gpu is None:
        use_gpu = gpu_renders()
    mi.set_variant("cuda_ad_rgb" if use_gpu else "scalar_rgb")
    print("ray tracing on the " + ("GPU" if use_gpu else "CPU (no usable GPU found)") + f": {pixels} x {pixels} pixels, {samples} samples per pixel")
    T = mi.ScalarTransform4f
    x = np.array(packing.positions, dtype=float)
    n = len(x)
    assert x.shape[1] == 3, "render_raytraced draws 3D packings"
    D = np.array(packing.diameters, dtype=float)
    shrink = getattr(packing, "shrink_factor", None)                 # the sizes at which no two particles overlap,
    if shrink is not None:                                           # when the package reports them
        D = D * shrink
    ratios = getattr(packing, "aspect_ratios", None)
    R = getattr(packing, "orientations", None)
    ellipsoids = bool(ratios) and R is not None and len(R) == n      # otherwise: spheres
    in_sphere = list(packing.walls)[0] == -3
    box = np.array(packing.box, dtype=float)
    if in_sphere:
        box = np.array([box[0]] * 3)
    c = 0.5 * box                                                    # look at the centre of the box
    # the radius of a sphere that holds everything drawn, and a camera far enough to see all of it
    reach = 0.504 * float(box[0]) if in_sphere else 0.5 * float(np.linalg.norm(box)) + 0.5 * float(D.max())
    view = np.array([1.55, -1.75, 0.95]) / np.linalg.norm([1.55, -1.75, 0.95])
    distance = 1.15 * reach / np.sin(np.radians(15.0))               # the field of view is 30 degrees
    scene = {
        "type": "scene",
        "integrator": {"type": "path", "max_depth": 6, "hide_emitters": True},
        "sensor": {"type": "perspective", "fov": 30,
                   "to_world": T().look_at(origin=(c + distance * view).tolist(), target=c.tolist(), up=[0, 0, 1]),
                   "film": {"type": "hdrfilm", "width": pixels, "height": pixels, "rfilter": {"type": "gaussian"}, "pixel_format": "rgba"},
                   "sampler": {"type": "independent", "sample_count": samples}},
        # even light from all round, and one distant lamp for shape
        "sky": {"type": "constant", "radiance": {"type": "rgb", "value": [0.78, 0.78, 0.78]}},
        "key": {"type": "directional", "direction": [-0.55, 0.65, -0.75], "irradiance": {"type": "rgb", "value": [1.1, 1.1, 1.1]}},
    }
    if in_sphere:          # the container: a faint clear shell
        scene["shell"] = {"type": "sphere", "center": c.tolist(), "radius": 0.504 * float(box[0]),
                          "bsdf": {"type": "thindielectric", "int_ior": 1.2, "ext_ior": 1.0}}
    elif ellipsoids:       # an invisible sphere round the box (a scene made only of stretched copies does not render)
        scene["bound"] = {"type": "sphere", "center": c.tolist(), "radius": reach, "bsdf": {"type": "null"}}
    for k, col in enumerate(CANDY):
        scene[f"candy{k}"] = {"type": "roughplastic", "distribution": "ggx", "alpha": 0.35, "int_ior": 1.3,
                              "diffuse_reflectance": {"type": "rgb", "value": [min(1.0, 1.08 * v + 0.02) for v in col]}}
    colour = np.random.default_rng(seed).integers(len(CANDY), size=n)
    if ellipsoids:
        # every ellipsoid is a copy of a unit sphere, stretched and turned: exact, no mesh
        for k in range(len(CANDY)):
            scene[f"unit{k}"] = {"type": "shapegroup", "s": {"type": "sphere", "bsdf": {"type": "ref", "id": f"candy{k}"}}}
        R = np.array(R, dtype=float)
        rows = np.array(ratios, dtype=float)
        rows = np.tile(rows, (n, 1)) if rows.ndim == 1 else rows      # one shape for all, or one per particle
        semi = 0.5 * D[:, None] * np.concatenate([np.ones((n, 1)), rows], axis=1)
        for i in range(n):
            M = np.eye(4)
            M[:3, :3] = R[i] @ np.diag(semi[i])
            M[:3, 3] = x[i]
            scene[f"p{i}"] = {"type": "instance", "to_world": T(M.tolist()), "g": {"type": "ref", "id": f"unit{int(colour[i])}"}}
    else:
        for i in range(n):
            scene[f"p{i}"] = {"type": "sphere", "center": x[i].tolist(), "radius": 0.5 * float(D[i]),
                              "bsdf": {"type": "ref", "id": f"candy{int(colour[i])}"}}
    t0 = time.time()
    if use_gpu:
        # the GPU traces all rays of a call at once, so the picture is the average of passes of about 8 million rays
        per_pass = max(1, min(samples, int(8e6 // (pixels * pixels))))
        passes = -(-samples // per_pass)
        loaded = mi.load_dict(scene)
        img = sum(np.array(mi.render(loaded, spp=per_pass, seed=k), dtype=np.float64) for k in range(passes)) / passes
    else:
        img = np.array(mi.render(mi.load_dict(scene)), dtype=np.float64)
    img = np.clip(img, 0.0, 1.0)
    rgb = img[:, :, :3]
    img[:, :, :3] = np.where(rgb <= 0.0031308, 12.92 * rgb, 1.055 * rgb ** (1 / 2.4) - 0.055)      # linear light -> sRGB
    picture = Image.fromarray((img[:, :, :4] * 255 + 0.5).astype(np.uint8), "RGBA")
    picture.save(path)
    print(f"ray tracing time: {time.time() - t0:.1f} s   (saved {path})")
    return picture


In [ ]:
candies = rcpgenerator.Packing(
    N=750, Ndim=3, box=[1.0, 1.0, 1.0],
    walls=[-3, 0, 0],                 # a spherical container
    aspect_ratios=[1.0, 0.5],         # oblate: two long axes, one of half the length
    phi=0.05, seed=1,
)
timed_pack(candies)
print(f"phi at contact: {candies.phi_contact:.4f}")

In [ ]:
# Ray trace the packing made in the previous cell
picture = render_raytraced(candies, pixels=600, samples=32)
picture

### Spheres in a Box

The same renderer draws spheres, and a periodic or walled box: it frames the whole box. Particles are drawn at the
sizes at which no two overlap (`shrink_factor`).

In [ ]:
marbles = rcpgenerator.Packing(
    N=1500, Ndim=3, box=[1.0, 1.0, 1.0], walls=[0, 0, 1],      # periodic in x and y, walls on z
    dist={"type": "powerlaw", "d_min": 1.0, "d_max": 4.0, "exponent": -3.0},
    phi=0.05, seed=1,
)
timed_pack(marbles)
print(f"phi at contact: {marbles.phi_contact:.4f}")

In [ ]:
# Ray trace the packing made in the previous cell
picture = render_raytraced(marbles, pixels=600, samples=32, path="spheres_raytraced.png")
picture